In [ ]:
%load_ext autoreload
%autoreload 2
# test

In [2]:
import sys
import os

# Get the current working directory of the notebook
current_notebook_dir = os.getcwd()

# Get the parent directory
parent_dir = os.path.abspath(os.path.join(current_notebook_dir, os.pardir))

# Add the parent directory to the Python path
sys.path.append(parent_dir)

In [3]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from torchvision import models
import matplotlib.pyplot as plt

from deepfake_utils.datasets import DeepFakeDataset
from deepfake_utils.train import train
from torchvision.models import resnet50, ResNet50_Weights, vit_b_32, ViT_B_32_Weights
import random
from sklearn.model_selection import train_test_split
from PIL import Image

In [4]:
SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
if torch.mps.is_available():
    torch.mps.manual_seed(SEED)

In [5]:
torch.set_default_dtype(torch.float32)

In [6]:
image_metadata_path = "../Deepfake-Eval-2024/image-metadata-publish.csv"
image_dir_path = '../Deepfake-Eval-2024/image-data'

In [7]:
device = torch.accelerator.current_accelerator()
print(f'Using {device} accelerator')

Using mps accelerator


In [8]:
# Hyperparameters
learning_rate = 1e-3
batch_size = 32
epochs = 3
dropout_rate = 0.2
num_classes = 2

In [9]:
# Loss function
loss_fn = nn.CrossEntropyLoss(reduction = 'sum')

In [10]:
# Model Architecture
weights = ViT_B_32_Weights.DEFAULT
model = vit_b_32(weights=weights)

# Freeze parameters within Vision Transformer
for param in model.parameters():
    param.requires_grad = False

# Replace last fully connected layer
num_ftrs = model.heads.head.in_features

# Add dropout and modify last linear layer
model.heads = nn.Sequential(
    nn.Dropout(dropout_rate),
    nn.Linear(num_ftrs, num_classes)
)

model.to(device)

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [11]:
# Optimizer and learning rate schedule
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=1, eta_min=1e-6, last_epoch=-1
)

In [12]:
# Load data
train_data = DeepFakeDataset(os.path.join(parent_dir, "image-metadata-train.csv"), image_dir_path, 'ViT', is_train = True)
train_data_loader = DataLoader(train_data, batch_size = batch_size, shuffle = True)

val_data = DeepFakeDataset(os.path.join(parent_dir, "image-metadata-val.csv"), image_dir_path, 'ViT', is_train = False)
val_data_loader = DataLoader(val_data, batch_size = batch_size, shuffle = False)

In [13]:
experiment_id = 1

In [17]:
# Train model
train_loss_history, train_acc_history, val_loss_history, val_acc_history = train(
    epochs,
    train_data_loader,
    val_data_loader,
    model,
    loss_fn,
    optimizer,
    device,
    lr_scheduler=lr_scheduler
    )
experiment_record = {
    'experiment_id': experiment_id,
    'model': 'ViT',
    'train_loss_history': [train_loss_history], 
    'train_acc_history': [train_acc_history], 
    'val_loss_history': [val_loss_history], 
    'val_acc_history': [val_acc_history],
    'train_loss': [train_loss_history[-1]], 
    'train_acc': [train_acc_history[-1]], 
    'val_loss': [val_loss_history[-1]], 
    'val_acc': [val_acc_history[-1]],
}

Epoch 1
- - - - - - - - - - - - - - - - - - - - - - - - - 
Training...
	Training Progress: 	[   32/ 1365]
	Training Progress: 	[   64/ 1365]
	Training Progress: 	[   96/ 1365]
	Training Progress: 	[  128/ 1365]
	Training Progress: 	[  160/ 1365]
	Training Progress: 	[  192/ 1365]
	Training Progress: 	[  224/ 1365]
	Training Progress: 	[  256/ 1365]
	Training Progress: 	[  288/ 1365]
	Training Progress: 	[  320/ 1365]
	Training Progress: 	[  352/ 1365]
	Training Progress: 	[  384/ 1365]
	Training Progress: 	[  416/ 1365]
	Training Progress: 	[  448/ 1365]
	Training Progress: 	[  480/ 1365]
	Training Progress: 	[  512/ 1365]
	Training Progress: 	[  544/ 1365]
	Training Progress: 	[  576/ 1365]
	Training Progress: 	[  608/ 1365]
	Training Progress: 	[  640/ 1365]
	Training Progress: 	[  672/ 1365]
	Training Progress: 	[  704/ 1365]
	Training Progress: 	[  736/ 1365]
	Training Progress: 	[  768/ 1365]
	Training Progress: 	[  800/ 1365]
	Training Progress: 	[  832/ 1365]
	Training Progress:

In [18]:
experiment_record

{'experiment_id': 1,
 'model': 'ViT',
 'train_loss_history': [[0.4616386092189467,
   0.4467565648285024,
   0.4375809065151564]],
 'train_acc_history': [[0.7846153974533081,
   0.7941392064094543,
   0.8065934181213379]],
 'val_loss_history': [[0.537348651314435,
   0.5460824839872856,
   0.5396453415694302]],
 'val_acc_history': [[0.715753436088562,
   0.7260273694992065,
   0.6917808055877686]],
 'train_loss': [0.4375809065151564],
 'train_acc': [0.8065934181213379],
 'val_loss': [0.5396453415694302],
 'val_acc': [0.6917808055877686]}

In [ ]:
# record and experiment results and save to csv
output_filename = "experiment_results.csv"
pd.DataFrame(experiment_record).to_csv(output_filename, mode = 'a', header = os.path.exists(output_filename) == False, index = False)
with open(f"experiment_{experiment_id}.pth", "wb") as file:
    torch.save(model.state_dict(), file)